# Evaluation foundations: learn to judge a summary before trusting a score

**Offline • fictional source • explicitly authored teaching labels**

You will distinguish a faithful summary from a complete one, calculate a small support ratio,
inspect real quotation and structure checks, and decide what a human still needs to review.
No credentials, language-model requests, or externally verified financial facts are used.

The worked labels below were written for this lesson. They are not predictions from an AI
judge, independent subject-matter calibration, or evidence about real meetings.

## Goal

By the end, explain these five things in your own words:

1. Why a summary can be entirely faithful and still be unhelpful.
2. Why one important error can matter despite a high average score.
3. What an exact quotation check proves and what it does not.
4. Why a failed evaluation must not become a zero summary score.
5. Which prompt tells Zoom what to write and which prompts tell a judge how to evaluate it.

Read a prediction question before running its next code cell. Change one example after you
understand the worked answer; explain your expected result before looking at the output.

## Setup

Install `.[dev,notebooks]` from the project root and use that Python environment as your kernel.
This lesson does not load `.env` or construct a model client. The package is imported first
so its DeepEval privacy settings apply. The default tutorial runner also blocks outgoing
Python socket connections.

If an import fails, compare the printed Python path with the project's `.venv/bin/python`.

In [ ]:
# Check the active kernel before importing the evaluator or its dependencies.
import sys
import tomllib
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file() and (path / "src/meeting_eval").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the meeting-summary-eval repository or its notebooks folder.")
dependencies = tomllib.loads((ROOT / "pyproject.toml").read_text(encoding="utf-8"))["project"]["dependencies"]
required_deepeval = next(item.removeprefix("deepeval==") for item in dependencies if item.startswith("deepeval=="))
try:
    installed_deepeval = version("deepeval")
except PackageNotFoundError:
    installed_deepeval = "not installed"
print("Python:", sys.executable)
print("DeepEval:", installed_deepeval, "| Required:", required_deepeval)
if installed_deepeval != required_deepeval:
    raise RuntimeError(
        f"This kernel needs deepeval=={required_deepeval}; found {installed_deepeval}. "
        "From the repository root, run %pip install --upgrade -e '.[dev,notebooks]' "
        "to install into the active kernel, then restart the kernel and rerun from the top."
    )


In [ ]:
from pathlib import Path
import sys
import math
from IPython.display import Markdown, display
import meeting_eval
from meeting_eval.diagnostics import check_structure, verify_evidence
from meeting_eval.models import Evidence, Finding, MeetingResult, MetricResult
from meeting_eval.reports import score_rows

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
print("Python:", sys.executable)
print("Mode: offline; no model or credential setup")

def table(headers, rows):
    def cell(value):
        return str(value).replace("|", r"\|").replace("\n", " ")
    text = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    text.extend("| " + " | ".join(cell(value) for value in row) + " |" for row in rows)
    display(Markdown("\n".join(text)))

## 1. Know what each component does

| Component | Its job | In this repository |
|---|---|---|
| Producer | Write the summary from supplied sources. | An external summarizer supplied by your workflow. |
| Evaluation policy | Define what counts as supported, important, or material. | Shared policy and metric/diagnostic instructions. |
| Judge | Interpret source and summary using that policy. | A configured language model behind the adapter. |
| Metric | Turn judgments into a defined measurement. | Faithfulness ratio or rubric-based coverage. |
| Deterministic check | Verify an objective property. | Exact quotation matches and Markdown structure. |
| Human reviewer | Resolve ambiguity and test whether the evaluator is useful. | Independent labels and calibration. |

The source, summary, and policy are inputs to evaluation. A judge response is not ground
truth. A perfectly valid structured response can still contain a bad interpretation.
Our evaluator checks response completeness as well as preserving evidence for review.

## 2. Read the entire fictional source

The transcript establishes what was said. It does not prove an economic claim true in the
outside world. Additional context can establish its own supplied facts, but cannot make a
statement into something said in the meeting.

**Predict:** Which details would be misleading to change? Which would matter if omitted?

In [ ]:
TRANSCRIPT = """FICTIONAL TEACHING SCENARIO
Alex: Short-term yields fell 10 basis points this week.
Alex: Longer-term yields were unchanged.
Jordan: No date for a rate cut was forecast.
Sam: A higher yield does not eliminate price risk.
Sam: Ask advisors about near-term cash needs before discussing rate volatility.
"""
print(TRANSCRIPT)
IMPORTANT_POINTS = {
    "short_move": "Short-term yields fell 10 basis points this week.",
    "long_move": "Longer-term yields were unchanged.",
    "no_forecast": "No date for a rate cut was forecast.",
    "risk": "A higher yield does not eliminate price risk.",
    "liquidity": "Sam proposed asking about near-term cash needs.",
}
table(["Authored source point", "Meaning to retain"], IMPORTANT_POINTS.items())

## 3. Predict accuracy and omissions separately

These are short excerpts to isolate meaning; full formatting comes later. Decide whether
each included claim is supported, then separately ask which important source points are
missing. Do not use wording similarity as a substitute for meaning.

In [ ]:
EXCERPTS = {
    "Faithful but thin": "Short-term yields fell 10 basis points this week.",
    "Balanced": (
        "Short-term yields fell 10 basis points this week while longer-term yields were unchanged. "
        "No rate-cut date was forecast. Higher yield does not eliminate price risk. "
        "Sam proposed asking advisors about near-term cash needs before rate volatility."
    ),
    "Polished but wrong": (
        "Short-term yields rose 10 basis points this week. Longer-term yields were unchanged. "
        "No rate-cut date was forecast."
    ),
}
table(["Candidate excerpt", "Read before scoring"], EXCERPTS.items())

### Reveal the authored labels and calculate

We supply the claim boundaries and labels ourselves. The live pipeline asks a model to
extract and judge those claims, which introduces additional opportunities for mistakes.
The checklist fraction below treats the five teaching points equally; **it is not this
package's holistic GEval coverage score** and is not a universal definition of importance.

In [ ]:
CLAIM_LABELS = {
    "Faithful but thin": [("Weekly short-term decline", "yes")],
    "Balanced": [(point, "yes") for point in IMPORTANT_POINTS],
    "Polished but wrong": [("Weekly short-term rise", "no"),
                           ("Longer-term yields unchanged", "yes"),
                           ("No rate-cut date forecast", "yes")],
}
COVERED_POINTS = {
    "Faithful but thin": {"short_move"},
    "Balanced": set(IMPORTANT_POINTS),
    "Polished but wrong": {"long_move", "no_forecast"},
}
worked_rows = []
for name, labels in CLAIM_LABELS.items():
    supported = sum(verdict == "yes" for _, verdict in labels)
    faithfulness = supported / len(labels)
    checklist = len(COVERED_POINTS[name]) / len(IMPORTANT_POINTS)
    worked_rows.append([name, f"{supported}/{len(labels)}", f"{faithfulness:.3f}",
                        f"{len(COVERED_POINTS[name])}/5", f"{checklist:.3f}"])
table(["Excerpt", "Supported / claims", "Toy support ratio", "Covered / source points", "Toy checklist fraction"], worked_rows)
assert len(COVERED_POINTS["Faithful but thin"]) == 1
assert math.isclose(2 / len(CLAIM_LABELS["Polished but wrong"]), 2 / 3)
print("The thin excerpt has support ratio 1.0 while retaining only one of five teaching points.")

## 4. Learn the actual measurements

- **Faithfulness:** of the extracted claims, how many are supported? Under this package's
  conservative policy, borderline support does not count as a supported claim.
- **Coverage:** does important meaning survive at useful compression? Our GEval rubric
  emits a 0–10 judgment normalized to 0–1. A score of 0.8 does not mean 80% of sentences
  or checklist items survived.
- **Severity:** could a mistake materially change an advisor or client conversation?
  It remains separate from the ratio.
- **Structure and clarity:** is the summary organized as requested and easy to read?
  Neither can establish that it is accurate or complete.

**Predict:** How reassuring is a ratio of 99 supported claims out of 100 if the remaining
claim invents an institutional guarantee?

In [ ]:
supported_claims = 99
all_claims = 100
ratio = supported_claims / all_claims
material_issue = "The remaining claim invents a guarantee."
print(f"Illustrative ratio: {ratio:.2f}")
print("Independent material issue:", material_issue)
print("A high ratio does not cancel that issue. There is no weighted overall score or acceptance gate here.")

## 5. Verify quotations, then assess their meaning

Below is a deliberately incorrect excerpt and an authored finding about the reversed
market direction. The real verifier checks that each quoted substring uniquely occurs
in the indicated source and derives line locations itself. It does not ask an AI to invent
a line number.

**Predict:** Would `verified=True` prove the finding's interpretation correct?

In [ ]:
finding = Finding(
    kind="contradiction", severity="high", description="The summary reverses the short-term yield move.",
    evidence=[Evidence(source="transcript", quote="Short-term yields fell 10 basis points this week."),
              Evidence(source="summary", quote="Short-term yields rose 10 basis points this week.")],
)
verified = verify_evidence([finding], TRANSCRIPT, EXCERPTS["Polished but wrong"])[0]
table(["Source", "Quote", "Uniquely matched", "Derived line"], [
    [item.source, item.quote, item.verified, item.line_start] for item in verified.evidence
])
assert all(item.verified for item in verified.evidence)
print("The two passages occur where indicated; you still decide whether they establish the claimed reversal.")

misinterpreted = Finding(kind="uncertainty", severity="medium",
    description="Intentionally wrong teaching interpretation: the quote promises risk-free returns.",
    evidence=[Evidence(source="transcript", quote="A higher yield does not eliminate price risk.")])
verify_evidence([misinterpreted], TRANSCRIPT, EXCERPTS["Balanced"])
print("Even this intentionally wrong interpretation has a matched quote:", misinterpreted.evidence[0].verified)
print("Read the quotation: it warns about risk; it does not promise risk-free returns.")

## 6. Separate formatting from quality

This complete teaching summary uses the six current sections, explicit outreach provenance,
and the absent-Q&A marker. We supply `qa_present=False` ourselves because the tiny source
has no questions. In a real evaluation the diagnostic judge supplies that assessment;
uncertainty remains a review flag.

**Predict:** Could the formatting still pass if the first fact were reversed?

In [ ]:
FULL_SUMMARY = """## Executive Briefing
- Short-term yields fell 10 basis points this week while longer-term yields were unchanged.
- No rate-cut date was forecast, and higher yield does not eliminate price risk.

## Economy & Market Updates
- Rates: The short-term decline applied to this week; longer-term yields were unchanged.

## Advisor Outreach & Sales Actions
- Discussed action: Sam proposed asking about near-term cash needs before rate volatility.

## Market & Institutional Perspectives
- Outlook: No date for a rate cut was forecast.

## Questions and Answers
No substantive Q&A occurred.

## Search Tags
Interest Rates, Duration Risk, Liquidity
"""
structure, _ = check_structure(FULL_SUMMARY, qa_present=False)
wrong_structure, _ = check_structure(FULL_SUMMARY.replace("yields fell", "yields rose"), qa_present=False)
print("Original structure score:", structure.score)
print("Structure after reversing the fact:", wrong_structure.score)
assert structure.score == wrong_structure.score == 1.0
print("Only the factual meaning changed. Structure checks do not measure that meaning.")

## 7. Distinguish execution, quality scores, and review status

The following **constructed result objects** demonstrate reporting states. They were not
returned by a judge. A completed evaluation can flag a poor summary. A timeout yields a
missing score, not a zero that would falsely describe summary quality. A zero score is
still a valid measured result when the metric completed and judged no extracted claim
supported.

In [ ]:
examples = [
    MeetingResult(meeting_id="completed_with_issue", execution_status="completed",
        metrics={"faithfulness": MetricResult(name="faithfulness", score=2/3)},
        review_required=True, review_reasons=["An authored material issue requires inspection."],
        metadata={"diagnostics_status": "completed"}),
    MeetingResult(meeting_id="judge_timeout", execution_status="partial",
        metrics={"faithfulness": MetricResult(name="faithfulness", status="error", reason="Controlled timeout")},
        errors=["Controlled timeout"], review_required=True,
        metadata={"diagnostics_status": "completed"}),
    MeetingResult(meeting_id="completed_zero", execution_status="completed",
        metrics={"faithfulness": MetricResult(name="faithfulness", score=0.0)},
        review_required=True, metadata={"diagnostics_status": "completed"}),
]
rows = score_rows(examples)
table(["Constructed case", "Execution", "Score", "Review"], [
    [row["meeting_id"], row["execution_status"], row["faithfulness_score"], row["review_required"]]
    for row in rows
])
assert rows[1]["faithfulness_score"] is None
assert rows[2]["faithfulness_score"] == 0.0

## 8. Find the evaluation instructions

| You want to change… | Review… |
|---|---|
| What support and attribution mean | Shared policy. |
| How claims are extracted or coverage is scored | Metric templates, evaluation steps, and rubrics. |
| Which omissions/severity findings are returned | Diagnostic instructions. |

Compare supplied summary variants under the same frozen judge policy. Changing the
summaries and judging rules simultaneously makes it hard to know why the score moved.
An exported judge prompt bundle is a review copy; editing that copy does not change
runtime behavior.

In [ ]:
paths = {
    "Shared policy": ROOT / "src/meeting_eval/policy.py",
    "Metrics": ROOT / "src/meeting_eval/metrics.py",
    "Diagnostics": ROOT / "src/meeting_eval/diagnostics.py",
}
for role, path in paths.items():
    assert path.is_file()
    print(role + ":", path.resolve())
print("Offline prompt browser: meeting-eval-prompts --show policy")
print("Offline review bundle: meeting-eval-prompts --output results/prompt-review-001")

## Checks and practice

Before moving on, answer without looking back:

1. Why does the one-sentence excerpt have perfect support but weak coverage?
2. Which part of the score machinery chose the claim boundaries in this lesson?
3. What would change if a borderline claim counted as supported?
4. Why is the timeout different from the completed zero?
5. Why can a perfectly matched quotation accompany a wrong interpretation?
6. Why should an invented guarantee remain visible despite a high ratio?

Try changing the reversed figure in `EXCERPTS`, then update the authored claim labels
and source-point mapping yourself. The labels will **not** adapt automatically: there is
no model in this notebook. In the next lab a scripted teaching judge similarly makes
DeepEval's mechanics observable without claiming semantic accuracy.

## Next steps

1. Open [03 — DeepEval metric lab](03_deepeval_metric_lab.ipynb) to inspect the real metric
   classes, schemas, rendered prompts, and optional assessment questions.
2. Use [01 — Single meeting](01_single_meeting.ipynb) for your first deliberate live judge
   run, then read the claims, reasons, and evidence rather than only the numbers.
3. Use [04 — Prompt review and experiments](04_prompt_review_and_experiments.ipynb) to
   inspect judge instructions and compare supplied summaries with the evaluator fixed.

For practical use, independently label several representative real cases before looking
at model scores, inspect false positives and missed material errors, repeat noisy cases,
and keep a held-out set. Those steps calibrate the evaluator; this fictional tutorial does not.
Read the [learning guide](../docs/learning-guide.md) and [calibration workflow](../docs/calibration.md).